# Chapter 4: Probability & Statistics
## Notebook 02 - Intermediate

Key probability distributions, Central Limit Theorem, Bayes' theorem, and Maximum Likelihood Estimation—all with ML-relevant examples.

**What you'll learn:**
- Bernoulli, Binomial, Normal, Poisson, Uniform distributions
- Central Limit Theorem: why normal appears everywhere
- Bayes' theorem: medical test & spam classification
- Expected value, variance, MLE intuition

**Time estimate:** 3 hours

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*

## 1. Probability Distributions Overview

| Distribution | Use Case | Parameters |
|--------------|----------|------------|
| Bernoulli | Single binary trial (correct/incorrect) | p |
| Binomial | n Bernoulli trials (accuracy over n samples) | n, p |
| Normal | Continuous, CLT, activations | μ, σ² |
| Poisson | Counts per interval (events, requests) | λ |
| Uniform | Equal probability over range | a, b |

See `assets/diagrams/probability_distributions.svg` for relationships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial, exp, sqrt, pi

def binomial_pmf(k, n, p):
    """P(X=k) for Binomial(n,p). From scratch, no scipy."""
    if k < 0 or k > n:
        return 0.0
    coef = factorial(n) / (factorial(k) * factorial(n - k))
    return coef * (p ** k) * ((1 - p) ** (n - k))

def normal_pdf(x, mu=0, sigma=1):
    """Normal PDF from scratch. φ(x) = (1/√(2πσ²)) exp(-(x-μ)²/(2σ²))"""
    return (1 / (sigma * sqrt(2 * pi))) * exp(-0.5 * ((x - mu) / sigma) ** 2)

x_binom = np.arange(0, 21)
pmf_binom = [binomial_pmf(k, 20, 0.5) for k in x_binom]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Bernoulli
axes[0,0].bar([0, 1], [0.4, 0.6], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0,0].set_xlabel('Outcome')
axes[0,0].set_ylabel('P(X)')
axes[0,0].set_title('Bernoulli(p=0.6)')

# Binomial
axes[0,1].bar(x_binom, pmf_binom, color='steelblue', edgecolor='navy')
axes[0,1].set_xlabel('k (successes)')
axes[0,1].set_ylabel('P(X=k)')
axes[0,1].set_title('Bin(20, 0.5)')

# Normal
x_norm = np.linspace(-4, 4, 200)
y_norm = [normal_pdf(x) for x in x_norm]
axes[1,0].plot(x_norm, y_norm, 'b-', linewidth=2)
axes[1,0].fill_between(x_norm, y_norm, alpha=0.3)
axes[1,0].set_xlabel('x')
axes[1,0].set_ylabel('φ(x)')
axes[1,0].set_title('Normal(0, 1)')

# Poisson (approximation)
from scipy.stats import poisson
x_poiss = np.arange(0, 15)
pmf_poiss = poisson.pmf(x_poiss, 4)
axes[1,1].bar(x_poiss, pmf_poiss, color='coral', edgecolor='darkred')
axes[1,1].set_xlabel('k')
axes[1,1].set_ylabel('P(X=k)')
axes[1,1].set_title('Poisson(λ=4)')

plt.tight_layout()
plt.savefig('../assets/diagrams/distributions_overview.svg')
plt.show()

## 2. Central Limit Theorem

**Predict**: If we repeatedly sample means of 30 random values from a *Uniform* distribution, what shape will the distribution of those means have?

CLT: The mean of i.i.d. samples approaches Normal as n → ∞, regardless of the original distribution!

In [ ]:
# Simulate CLT: sample means from Uniform(0,1)
n_samples = 10000  # number of sample means
sample_size = 30   # each mean is from 30 draws

means = [np.random.uniform(0, 1, sample_size).mean() for _ in range(n_samples)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Original: single uniform sample
single_sample = np.random.uniform(0, 1, 1000)
axes[0].hist(single_sample, bins=30, color='steelblue', edgecolor='black', density=True)
axes[0].set_title('Uniform(0,1) — Original')
axes[0].set_xlabel('x')

# Sample means with n=5
means_5 = [np.random.uniform(0, 1, 5).mean() for _ in range(n_samples)]
axes[1].hist(means_5, bins=40, color='green', alpha=0.6, edgecolor='black', density=True, label='n=5')
axes[1].set_title('Means of 5 samples')
axes[1].set_xlabel('Sample mean')

# Sample means with n=30
axes[2].hist(means, bins=50, color='purple', alpha=0.7, edgecolor='black', density=True, label='n=30')
x_fit = np.linspace(0.3, 0.7, 100)
y_fit = [normal_pdf(x, 0.5, sqrt(1/12/30)) for x in x_fit]  # σ² = 1/12 for Uniform
axes[2].plot(x_fit, y_fit, 'r-', linewidth=2, label='Normal fit')
axes[2].set_title('Means of 30 samples → Normal!')
axes[2].set_xlabel('Sample mean')
axes[2].legend()

plt.tight_layout()
plt.savefig('../assets/diagrams/clt_simulation.svg')
plt.show()
print("As n increases, distribution of sample means becomes Normal (CLT).")

## 3. Bayes' Theorem

$$P(H|E) = \frac{P(E|H) \cdot P(H)}{P(E)}$$

- **Prior** P(H): belief before evidence
- **Likelihood** P(E|H): how likely evidence given hypothesis
- **Posterior** P(H|E): updated belief after evidence

See `assets/diagrams/bayes_theorem.svg` for the medical test example.

In [ ]:
def bayes_update(p_prior, p_likelihood, p_evidence):
    """Posterior = Likelihood * Prior / Evidence"""
    return (p_likelihood * p_prior) / p_evidence if p_evidence > 0 else 0.0

# Medical test: P(disease)=0.01, sensitivity=0.95, specificity=0.98
p_disease = 0.01
p_pos_given_disease = 0.95  # sensitivity
p_neg_given_healthy = 0.98  # specificity
p_pos_given_healthy = 1 - p_neg_given_healthy

p_pos = p_pos_given_disease * p_disease + p_pos_given_healthy * (1 - p_disease)
p_disease_given_pos = bayes_update(p_disease, p_pos_given_disease, p_pos)

print("Medical test example:")
print(f"  P(disease) = {p_disease}")
print(f"  P(positive|disease) = {p_pos_given_disease}")
print(f"  P(disease|positive) = {p_disease_given_pos:.3f}")
print("→ Even with positive test, only ~32% chance of disease (rare prior).")

In [ ]:
# Spam classification: P(spam)=0.4, P('free'|spam)=0.6, P('free'|ham)=0.1
p_spam = 0.4
p_free_given_spam = 0.6
p_free_given_ham = 0.1
p_free = p_free_given_spam * p_spam + p_free_given_ham * (1 - p_spam)
p_spam_given_free = bayes_update(p_spam, p_free_given_spam, p_free)

print("\nSpam classification example:")
print(f"  P(spam) = {p_spam}")
print(f"  P('free'|spam) = {p_free_given_spam}")
print(f"  P(spam|'free') = {p_spam_given_free:.3f}")
print("→ Word 'free' raises spam probability significantly.")

## 4. Expected Value and Variance

- **E[X]**: long-run average. For model accuracy: E[accuracy] over many runs.
- **Var(X)**: spread. For portfolio: lower variance = more stable returns.

**ML example**: Two models A and B. A has higher mean accuracy but higher variance. Under limited evaluation budget, B might be preferred (more reliable).

In [ ]:
# Simulated model accuracies over 100 runs (e.g., different random seeds)
np.random.seed(42)
model_a = np.random.normal(0.92, 0.05, 100)  # higher mean, higher variance
model_b = np.random.normal(0.88, 0.02, 100)  # lower mean, lower variance

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([model_a, model_b], labels=['Model A (high var)', 'Model B (low var)'])
ax.set_ylabel('Accuracy')
ax.set_title('Model Selection: Mean vs Variance (100 runs each)')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../assets/diagrams/expected_value_variance.svg')
plt.show()
print(f"Model A: E[X]={model_a.mean():.3f}, Var={model_a.var():.4f}")
print(f"Model B: E[X]={model_b.mean():.3f}, Var={model_b.var():.4f}")

## 5. Maximum Likelihood Estimation (MLE) Intuition

**MLE**: Choose parameters that make the observed data most probable.

For Bernoulli: given data [1,0,1,1,0], MLE of p = proportion of 1s = 3/5 = 0.6.

**Predict**: If we observe 7 heads in 10 flips, what is the MLE of p?

In [ ]:
def mle_bernoulli(data):
    """MLE for Bernoulli: p_hat = proportion of successes."""
    return sum(data) / len(data) if data else 0.0

data = [1, 0, 1, 1, 0, 1, 1, 1, 0, 1]  # 7 ones
p_mle = mle_bernoulli(data)
print(f"Data: {data}")
print(f"MLE p_hat = {p_mle}")

# Visualize likelihood as function of p
p_vals = np.linspace(0.01, 0.99, 100)
likelihoods = [np.prod([p if x==1 else (1-p) for x in data]) for p in p_vals]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_vals, likelihoods, 'b-')
ax.axvline(x=p_mle, color='red', linestyle='--', label=f'MLE p={p_mle}')
ax.set_xlabel('p')
ax.set_ylabel('Likelihood L(p|data)')
ax.set_title('Bernoulli MLE: Likelihood maximized at p = 7/10')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/diagrams/mle_bernoulli.svg')
plt.show()

## 6. Summary

- **Distributions**: Bernoulli, Binomial, Normal, Poisson, Uniform—all with ML use cases
- **CLT**: Sample means → Normal as n increases
- **Bayes**: Update beliefs with evidence (medical test, spam)
- **E[X], Var(X)**: Mean and spread matter for model selection
- **MLE**: Choose parameters that maximize likelihood of observed data

Next: Hypothesis testing, confidence intervals, A/B testing.

---
*Generated by Berta AI | Created by Luigi Pascal Rondanini*